In [3]:
# Čia yra Python skriptas, skirtas EKG triukšmo aptikimui naudojant iš anksto apmokytą U-Net modelį.
# Skriptas naudoja TensorFlow ir Keras bibliotekas, kad įkeltų modelį ir atliktų EKG signalų apdorojimą.
# Prieš tai dar surandamos išskirtys (outliers) ir rspragos (rdropouts), kurie yra pašalinami iš EKG signalo.
# Išvedamas grafikas su aptiktais U-Net triukšmais ir išsaugomas kaip paveikslėlis.  išskirtys (outliers)
# ir rspragos (rdropouts) nevaizduojami, nes jie jau buvo pašalinti iš signalo.

# https://grok.com/chat/68e9c626-abf9-4fc3-9102-d82468902b6f

# Pipeline:
# textraw ECG (.npy)
#    ↓
# bandpass filtered (0.5–40 Hz)
#    ↓
# normalized (zero mean, unit variance)   ← this is ecg_normalized
#    ↓
# cut into overlapping 1024-sample pieces
#    ↓
# U-Net prediction
#    ↓
# overlap-add + average → denoised_signal   (still normalized scale)

# denoised ECG = denoised_signal
# → the model's best estimate of what the clean ECG looked like in the normalized domain


import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from scipy.signal import butter, filtfilt
from pathlib import Path
import math, json
import logging
from datetime import datetime
from dataclasses import asdict
import os, sys
import shutil
from datetime import datetime


# === Išoriniai moduliai (lygiagretus aplankas) =================================
PARALLEL_PATH = Path().resolve().parent / "SUPL_FUNCTIONS"
sys.path.append(str(PARALLEL_PATH))

from zive_util_ml import get_ecg_signal, get_ecg_noise_indices_annotated
from zive_util_ml import divide_signal_into_fragments, plot_signal_write_plot_R_P
from use_ecg_denoising_util import find_outliers_rdropouts
from unet_model_util import get_unet_model
from use_ecg_denoising_util import ecg_filter
from project_util import find_project_root_by_name


def show_lr_and_layer_names(model) -> float | None:
    """
    Prints the optimizer learning rate (if compiled) and all layer names.
    Returns the numeric LR if it can be resolved, otherwise None.
    Compatible with TF 2.17 / Python 3.11.
    """
    lr_val = None
    opt = getattr(model, "optimizer", None)
    if opt is None:
        print("Learning rate: (model not compiled)")
    else:
        lr = getattr(opt, "learning_rate", getattr(opt, "lr", None))
        try:
            if isinstance(lr, (int, float)):
                lr_val = float(lr)
            elif isinstance(lr, tf.keras.optimizers.schedules.LearningRateSchedule):
                lr_val = float(lr(tf.constant(0, dtype=tf.int64)).numpy())
            elif callable(lr):  # custom callable schedule
                lr_val = float(tf.convert_to_tensor(lr(0)).numpy())
            elif hasattr(lr, "numpy"):  # tf.Variable / tf.Tensor
                lr_val = float(lr.numpy())
            else:
                lr_val = float(lr)
            print("Learning rate:", lr_val)
        except Exception:
            print("Learning rate: (unresolved)", lr)

    print("Layers:")
    for layer in model.layers:
        print(" -", layer.name)

    return lr_val


# Normalization function (matched to training script)
def normalize(signal):
    logger.debug("Normalizing signal")
    mean = np.mean(signal)
    std = np.std(signal)
    if std < 1e-8:
        logger.warning("Signal has near-zero variance; returning zeros")
        return np.zeros_like(signal)
    return (signal - mean) / std

# Preprocessing function
def preprocess_ecg(ecg, segment_length, overlap, fs):
    logger.info("Preprocessing ECG signal")
    
    # Apply bandpass filter - užkomentuota, nes filtras jau buvo pritaikytas visam signalui jį nuskaičius
    # ecg_filtered = filter_ecg(ecg_data_orig, FILTER)
    # logger.debug(f"Bandpass filter applied, filtered signal length: {len(ecg_filtered)}")
    
    # Normalize the signal
    ecg_normalized = normalize(ecg)
    # ecg_normalized = normalize(ecg_filtered)
    logger.debug(f"Signal normalized, mean: {np.mean(ecg_normalized):.4f}, std: {np.std(ecg_normalized):.4f}")
    
    # Segment the ECG into overlapping windows
    step = int(segment_length * (1 - overlap))
    segments = []
    indices = []
    for start in range(0, len(ecg_normalized) - segment_length + 1, step):
        segment = ecg_normalized[start:start + segment_length]
        if len(segment) == segment_length:
            segments.append(segment)
            indices.append(start)
    segments = np.array(segments)
    logger.debug(f"Segmented into {len(segments)} segments, step size: {step}")
    
    # Reshape for U-Net input (shape: [batch, segment_length, 1])
    segments = segments[:, :, np.newaxis]
    logger.debug(f"Segments reshaped for model input, shape: {segments.shape}")
    return segments, indices, ecg_normalized

# Function to merge sticking or overlapping intervals
def merge_intervals(indices, segment_length):
    if not indices:
        logger.debug("No noisy indices to merge")
        return []
    # Convert segment start indices to intervals
    intervals = []
    for idx in sorted(indices):
        intervals.append([idx, idx + segment_length - 1])
    logger.debug(f"Created {len(intervals)} initial intervals")
    
    # Merge overlapping or adjacent intervals
    merged = []
    current = intervals[0]
    for next_interval in intervals[1:]:
        if next_interval[0] <= current[1] + 1:  # Overlapping or adjacent
            current[1] = max(current[1], next_interval[1])
        else:
            merged.append(current)
            current = next_interval
    merged.append(current)
    logger.debug(f"Merged into {len(merged)} intervals")
    return merged

# Postprocessing function to detect and merge noisy fragments
def detect_noisy_fragments(original, denoised, indices, segment_length, overlap, threshold=0.1):
    logger.info("Detecting noisy fragments")
    noisy_indices = []
    residual = np.abs(original - denoised)
    logger.debug(f"Residual computed, mean: {np.mean(residual):.4f}, std: {np.std(residual):.4f}")
    
    # Identify noisy segments
    for i, start_idx in enumerate(indices):
        segment_residual = residual[start_idx:start_idx + segment_length]
        mean_residual = np.mean(segment_residual)
        if mean_residual > threshold:
            noisy_indices.append(start_idx)
            logger.debug(f"Noisy segment detected at index {start_idx}, mean residual: {mean_residual:.4f}")
    
    logger.debug(f"Found {len(noisy_indices)} noisy segments")
       
    # Merge noisy intervals
    noisy_intervals = merge_intervals(noisy_indices, segment_length)
    return noisy_intervals


def run_ecg_denoising_pipeline(ecg_data, model, CONFIG, threshold):
    """
    Full ECG denoising pipeline using a trained model.

    Parameters:
    - ecg_data: np.ndarray, raw ECG signal
    - model: trained Keras model (e.g., U-Net)
    - CONFIG: dict, configuration parameters including:
        - FS: int, sampling frequency
        - SEGMENT_LENGTH: int, length of each segment for processing
        - OVERLAP: float, fraction of overlap between segments
    - threshold: float, noise detection threshold

    Returns:
    - denoised_signal: np.ndarray
    - noisy_intervals: list of (start, end) indices
    """
    
    fs = CONFIG['FS']
    segment_length = CONFIG['SEGMENT_LENGTH']
    overlap = CONFIG['OVERLAP']
    logger.info("Starting ECG denoising pipeline")
    logger.debug(f"Configuration: fs={fs}, segment_length={segment_length}, overlap={overlap}, threshold={threshold}")  
    logger.debug(f"ECG data shape: {ecg_data.shape}")
    
    # Step 1: Preprocess
    segments, segment_indices, ecg_normalized = preprocess_ecg(ecg_data, segment_length,  overlap, fs)

    # Step 2: Predict
    logger.info("Predicting denoised segments")
    denoised_segments = model.predict(segments)
    logger.debug(f"Prediction completed, denoised segments shape: {denoised_segments.shape}")

    # Step 3: Reconstruct full signal
    logger.info("Reconstructing denoised signal")
    denoised_signal = np.zeros_like(ecg_normalized)
    count = np.zeros_like(ecg_normalized)
    for i, start_idx in enumerate(segment_indices):
        end_idx = start_idx + segment_length
        denoised_signal[start_idx:end_idx] += denoised_segments[i, :, 0]
        count[start_idx:end_idx] += 1
    count[count == 0] = 1
    denoised_signal /= count
    logger.debug(f"Denoised signal reconstructed, length: {len(denoised_signal)}")

    # Step 4: Detect noisy intervals
    noisy_intervals = detect_noisy_fragments(ecg_normalized, denoised_signal, segment_indices, segment_length, overlap, threshold)
    noisy_intervals_secs = [(x/fs, y/fs) for x, y in noisy_intervals]

    print(f"\nNoisy intervals (start, end): {noisy_intervals}")
    print(f"Noisy intervals secs (start, end): {noisy_intervals_secs}")

    logger.debug(f"Noisy intervals detected: {noisy_intervals}")
    logger.debug(f"Noisy intervals detected secs: {noisy_intervals_secs}")
    logger.info("Noise detection completed")

    return denoised_signal, noisy_intervals


# --------------------------- CONFIGURATION -------------------------------------

PROJECT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = find_project_root_by_name(target="PROJECT_TRAIN_UNET", start=PROJECT_DIR)
print("\nPROJECT ROOT DIR:", PROJECT_ROOT)
print("PROJECT DIR:", PROJECT_DIR)

# Configure logging
log_dir = 'use_logs'

# If the log_dir exists, clean it
if os.path.exists(log_dir):
    shutil.rmtree(log_dir)

# Recreate the log_dir
os.makedirs(log_dir, exist_ok=True)

# Create the log file path
log_file = os.path.join(log_dir, f'ecg_noise_detection_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')


# Create logger and set level to DEBUG to capture all messages
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

# Clear any existing handlers to prevent interference
logger.handlers.clear()

# Suppress matplotlib debug messages
logging.getLogger('matplotlib').setLevel(logging.WARNING)
# logging.getLogger('tensorflow').setLevel(logging.WARNING)
# logging.getLogger('scipy').setLevel(logging.WARNING)

# Create file handler for writing DEBUG and above to log file
file_handler = logging.FileHandler(log_file)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s'))

# Create stream handler for writing INFO and above to console
stream_handler = logging.StreamHandler()
stream_handler.setLevel(logging.INFO)
stream_handler.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s'))

# Add handlers to logger
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

logger.info(f"Starting ECG noise detection script. Log file: {log_file}")



# Configuration parameters (matched to training script)
CONFIG = {
    'FS': 200,              # Sampling rate
    'SEGMENT_LENGTH': 1024, # Segment length
    'OVERLAP': 0.5          # Overlap fraction
}

# Log configuration parameters
logger.info(f"Configuration: {json.dumps(CONFIG, indent=2)}")

segment_length = CONFIG['SEGMENT_LENGTH']
overlap = CONFIG['OVERLAP']
threshold = 0.12  # Residual threshold for noise detection (tune as needed) , smaller - more sensitive
# threshold = 0.12  # Residual threshold for noise detection (tune as needed)
fs = CONFIG['FS']

print(f"\nthreshold: {threshold}")

# Load unet model
unet_model_dir = PROJECT_ROOT/'MODEL_UNET'
model_filename = 'resunet_ecg_1024_0_5_0_7_test.keras'

bundle = get_unet_model(unet_model_dir, model_filename, segment_length, logger)  # you can pass your logger or omit
model = bundle.model
# +++++++++++++++++++++
# _ = show_lr_and_layer_names(model)
# +++++++++++++++++++++
expected_input_shape = bundle.expected_input_shape
logger = bundle.logger  # <- here you go
logger.info(f"Loading U-Net model from {bundle.model_path}")

# bendras aplankas grafikams
MARKER = model_filename.removeprefix("resunet_ecg").removesuffix(".keras")  # -> "_1024_0_5_3_7"
plot_save_dir = "plottings" + MARKER

# --------------------------   MAIN CODE -------------------------------------

# Duomenų aplankas su zive ir mit duomenimis
Duomenu_aplankas = PROJECT_ROOT/'DATA_ORIG'

# Aplankas su EKG įrašais (.npy) ir anotacijomis (.json)
db_folder = 'ecg_zive_npy'  # zive 
# db_folder = 'Igno'  # Igno Griskeviciaus

# Nuoroda į aplanką su EKG įrašais (.npy) ir anotacijomis (.json)
rec_dir = Path(Duomenu_aplankas, db_folder)

fileName = '1009_6.npy' # 2222, geras, tik keletas vietų su triukšmais
fileName = '1002_6.npy' # 2222, daug triukšmų

fileName = '1001_6.npy' # Iš Žilvino


# Žilvino papildomi įrašai
fileNames = ['1001_6', '1102_0', '1001_7', '1001_8', '1103_0', '1001_9', '1102_1', '1001_10', '1001_11']
fileNames = [fileName + '.npy' for fileName in fileNames]

# Su triukšmais
fileNames = ['1001_2.npy'] # iš Žilvino, su triukšmais, tinka demonstracijai
fileNames = ['1002_1.npy'] #  Švarumas 6, triukšmai sužymėti,  įtrauktas į triukšmų testavimo sąrašą 
fileNames = ['1002_2.npy'] #  Švarumas 6, verta triukšmus anotuoti ir įtraukti į testavimo sąrašą 
fileNames = ['1006_2.npy'], # kj Švarumo lygis 8, triukšmas pažymėtas, daugiau tinka triukšmų testavimui
fileNames = ['1006_3.npy'] # kj Švarumo lygis 8, triukšmas pažymėtas, daugiau tinka triukšmų testavimui

fileNames = ['1001_2.npy', '1002_1.npy', '1002_2.npy', '1006_2.npy', '1006_3.npy'] 

# Švarūs įrašai
fileNames = ['1102_1.npy', '1001_8.npy', '1005_4.npy', '1006_1.npy', '1001_6.npy']

# start0

fileNames = ['1001_2.npy'] # iš Žilvino, su triukšmais, tinka demonstracijai
fileNames = ['1031_18.npy'] #Outliers Secs: (139.0, 179.3) Rdropouts Secs: (127.0, 640.0)
fileNames = ['1009_1.npy'] #


# Pseudo_annotated
fileNames = ['1019_118.npy'] #
fileNames = ['1001_4.npy'] #
fileNames = ['1005_2.npy'] #
fileNames = ['1008_1.npy'] #
fileNames = ['1008_10.npy'] #


# ----------------------------- CIKLAS PER fileNames     -------------------------------------

for fileName in fileNames:
    
# ---------------------------- DUOMENŲ PARUOŠIMAS -------------------------------------
    
    # Load the ECG data from .npy file
    filePath = os.path.join(rec_dir, fileName)             
    print(f"f\nFailas: {filePath}")
    logger.info("Loading ECG data from %s", fileName)
    
    try:
        ecg_data_orig = np.array(get_ecg_signal(filePath))
        # Load corresponding JSON file
        json_path = os.path.splitext(filePath)[0] + '.json'
        noise_indices_annotated = get_ecg_noise_indices_annotated(json_path) 
        logger.debug(f"ECG data loaded, shape: {ecg_data_orig.shape}")
    except Exception as e:
        logger.error(f"Failed to load ecg_record.npy: {e}")
        raise

    print(f"\nNoisy intervals annotated (start, end): {noise_indices_annotated}")
    noise_indices_annotated_secs = [(x/fs, y/fs) for x, y in noise_indices_annotated]  # Convert to seconds
    print(f"Noisy intervals annotated secs (start, end): {noise_indices_annotated_secs}")
    # np.save('noisy_intervals.npy', noisy_intervals)

    # Apply  filter
    fp = {  'type': 'bandpass',
                'method':'butterworth',
                'order':4,
                'sampling_rate':fs,
                'lowcut':0.5,
                'highcut':70 }
    
    ecg_data_start = ecg_filter(ecg_data_orig, fp)
    logger.debug(f"Filter applied, filtered signal length: {len(ecg_data_start)}")

    # Toliau bus naudojamas filtruotas signalas


# ------------------------------- NOISE DETECTION AND DENOISING -------------------------------------
    
        # 1 DALIS: IŠSKIRČIŲ (OUTLIERS) IR RSPRAGŲ (RDROPOUTS) PAIEŠKA IR ŠALINIMAS

    print("\nIŠSKIRČIŲ (OUTLIERS) PAIEŠKA IR ŠALINIMAS")
    ecg_data, outliers_indices, rdropouts_indices = find_outliers_rdropouts(ecg_data_start)
    # ecg_data - tai filtruotas signalas, kuriame pašalintos išskirtys ir rspragos


    
    # outliers_indices_secs = [(x/fs, y/fs) for x, y in outliers_indices]  # Convert to seconds
    # rdropouts_indices_secs = [(x/fs, y/fs) for x, y in rdropouts_indices]  # Convert to seconds

        # 2 DALIS: detektuojami likusieji triukšmai
     
    print("\nLIKUSIŲ TRIUKŠMŲ PAIEŠKA IR ŠALINIMAS")
    denoised_signal, noisy_intervals = run_ecg_denoising_pipeline(ecg_data, model, CONFIG, threshold)
    logger.debug(f"Denoised signal reconstructed, length: {len(denoised_signal)}")

    # Detect and merge noisy fragments
    print(f"\nNoisy intervals (start, end): {noisy_intervals}")
    logger.debug(f"Noisy intervals detected: {noisy_intervals}")
    noisy_intervals_secs = [(x/fs, y/fs) for x, y in noisy_intervals]  # Convert to seconds
    print(f"Noisy intervals secs (start, end): {noisy_intervals_secs}")
    logger.debug(f"Noisy intervals detected secs: {noisy_intervals_secs}")

    # Optional: Save denoised ECG for inspection
    # np.save('denoised_ecg.npy', denoised_signal)
    logger.info("Noise detection completed")


# ---------------------- PLOTTING -------------------------------------

    # Ploting entire signal with all found distortions: outliers, rdropouts and oscillations
        
    flag_plot_start_with_distortions_entire = True
    portion_length_in_secs = 10  # seconds
    portion_length = portion_length_in_secs * fs
        
    if flag_plot_start_with_distortions_entire:
    #     print("\nPloting entire signal with all found distortions: outliers, rdropouts and oscillations")
    #     print(f"len(filtered_signal): {len(filtered_signal)} len(ecg_signal) secs: {len(ecg_signal_start)/fs:.1f}")
        
    #     print("\noutliers_indices_start_secs:", outliers_indices_start_secs)
    #     print("rdropouts_indices_start_secs:", rdropouts_indices_start_secs)
    #     print("oscillations_indices_start_secs:", oscillations_indices_start_secs)

        # Extract the numerical part from the fileName
        base_name = os.path.basename(fileName)

        flag_plot_save = True

        addition_to_name = 'denoised'
        threshold_str = str( threshold).replace('.', '_')
        
        if flag_plot_save:
            plot_dir = os.path.join(plot_save_dir, base_name + '_plot_' + addition_to_name + '_' + threshold_str)
            if not os.path.exists(plot_dir):
                os.makedirs(plot_dir)
        else:
            plot_dir = None

        print(f"\nplot_dir: {plot_dir}")

        print(f"filtered_signal length: {len(denoised_signal)} ({len(denoised_signal)/fs:.1f} secs)")
        print(f"portion_length: {portion_length} ({portion_length_in_secs} secs)")
        print(f"portion_length_in_secs: {portion_length_in_secs} ({portion_length} samples)")
        print(f"fs: {fs} ({1/fs:.1f} secs)")


        # Sudalijame  ecg_signal_start į fragmentus
        show_frag_indices = divide_signal_into_fragments(denoised_signal, portion_length)
        print()
        print(f"show_frag_indices: {show_frag_indices}")
        show_frag_indices_secs = [(x/fs, math.floor(y/fs)) for x, y in show_frag_indices] # indexes
        print(f"show_frag_indices_secs: {show_frag_indices_secs}")
            
        num_fragment = 1
        for (show_frag_start_secs, show_frag_end_secs) in show_frag_indices_secs:
            print(f"\nFRAGMENT NR. {num_fragment}")
            print(f"plot_signal_from_in_secs: {show_frag_start_secs}")
            print(f"plot_signal_to_in_secs: {show_frag_end_secs}")
            if (show_frag_end_secs - show_frag_start_secs) > 0.5:
                plot_signal_write_plot_R_P(fileName, denoised_signal, fs, num_fragment,
                    show_frag_start_secs, show_frag_end_secs, portion_length_in_secs,
                        plot_dir, addition_to_name,recID=None,
                        gap1_indices_secs = noisy_intervals_secs, # red
                        gap2_indices_secs = [], # blue
                        gap3_indices_secs= [], # yellow
                        # gap3_indices_secs=noise_indices_annotated_secs,
                        mark_indices_secs=[],
                        # annot_df=annot_df,
                        annot_df=pd.DataFrame(),   # tuščias df,
                        rpeak_indices_secs=[], ppeak_indices_secs=[],
                        flag_secs=True)
                    

                # plot_signal_write_plot_R_P(fileName, ecg_signal, fs, num_fragment, plot_signal_from_in_secs, plot_signal_to_in_secs, portion_length_in_secs, 
                #                     plot_save_dir, save_mark, recID=None,
                #                     gap1_indices_secs=[], gap2_indices_secs=[],
                #                     gap3_indices_secs=[], mark_indices_secs=[],
                #                     annot_df=None, 
                #                     rpeak_indices_secs=[], ppeak_indices_secs=[],
                #                     flag_secs=True):

            num_fragment += 1



2026-02-25 13:13:40,745 [INFO] Starting ECG noise detection script. Log file: use_logs/ecg_noise_detection_20260225_131340.log
2026-02-25 13:13:40,753 [INFO] Configuration: {
  "FS": 200,
  "SEGMENT_LENGTH": 1024,
  "OVERLAP": 0.5
}
2026-02-25 13:13:40,761 [INFO] Preparing to load U-Net model from /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/MODEL_UNET/resunet_ecg_1024_0_5_0_7_test.keras
2026-02-25 13:13:40,766 [INFO] U-Net model ready: resunet_ecg_1024_0_5_0_7_test.keras, expected_input_shape=(1024, 1)
2026-02-25 13:13:40,772 [INFO] Loading U-Net model from /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/MODEL_UNET/resunet_ecg_1024_0_5_0_7_test.keras
2026-02-25 13:13:40,784 [INFO] Loading ECG data from 1008_10.npy



PROJECT ROOT DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET
PROJECT DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/0_SELECT_ZIVE_DATA_2023

threshold: 0.12
f
Failas: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/DATA_ORIG/ecg_zive_npy/1008_10.npy

Noisy intervals annotated (start, end): []
Noisy intervals annotated secs (start, end): []

IŠSKIRČIŲ (OUTLIERS) PAIEŠKA IR ŠALINIMAS


2026-02-25 13:13:41,385 [INFO] Starting ECG denoising pipeline
2026-02-25 13:13:41,388 [INFO] Preprocessing ECG signal
2026-02-25 13:13:41,408 [INFO] Predicting denoised segments



LIKUSIŲ TRIUKŠMŲ PAIEŠKA IR ŠALINIMAS
8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 674ms/step


2026-02-25 13:13:47,515 [INFO] Reconstructing denoised signal
2026-02-25 13:13:47,546 [INFO] Detecting noisy fragments
2026-02-25 13:13:47,615 [INFO] Noise detection completed
2026-02-25 13:13:47,624 [INFO] Noise detection completed



Noisy intervals (start, end): [[0, 2047], [4096, 5119], [6144, 8191], [10240, 11775], [12800, 13823], [15360, 17407], [19456, 23039], [52736, 54783], [55808, 61951], [62464, 68607], [69632, 72703], [79872, 82431], [87552, 92671], [93184, 96767], [97280, 99839], [101376, 102911], [105984, 107519]]
Noisy intervals secs (start, end): [(0.0, 10.235), (20.48, 25.595), (30.72, 40.955), (51.2, 58.875), (64.0, 69.115), (76.8, 87.035), (97.28, 115.195), (263.68, 273.915), (279.04, 309.755), (312.32, 343.035), (348.16, 363.515), (399.36, 412.155), (437.76, 463.355), (465.92, 483.835), (486.4, 499.195), (506.88, 514.555), (529.92, 537.595)]

Noisy intervals (start, end): [[0, 2047], [4096, 5119], [6144, 8191], [10240, 11775], [12800, 13823], [15360, 17407], [19456, 23039], [52736, 54783], [55808, 61951], [62464, 68607], [69632, 72703], [79872, 82431], [87552, 92671], [93184, 96767], [97280, 99839], [101376, 102911], [105984, 107519]]
Noisy intervals secs (start, end): [(0.0, 10.235), (20.48, 25.